In [0]:
# caminho Edu
cadastro = spark.read.parquet("/Volumes/workspace/hackathon_2025/default/source/base_dados_cadastrais/part-00000-8b686964-78fc-4d3e-8888-86eb4f12cb6b-c000.snappy.parquet")

In [0]:
# caminho Michael
cadastro = spark.read.parquet("/Volumes/hackathon_2025/default/source/base_dados_cadastrais/")

In [0]:
display(cadastro)

In [0]:
cadastro.createOrReplaceTempView("cadastro")

#Select Distinct

In [0]:
%sql
SELECT DISTINCT FLAG_INSTALACAO FROM cadastro

In [0]:
%sql
SELECT DISTINCT FPD FROM cadastro

In [0]:
%sql
SELECT DISTINCT PROD FROM cadastro

In [0]:
%sql
SELECT DISTINCT flag_mig2 FROM cadastro

In [0]:
%sql
SELECT DISTINCT STATUSRF FROM cadastro

In [0]:
%sql
SELECT Safra,  
      COUNT(*)
        
FROM Cadastro
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT Safra,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT 
      FPD,  
      COUNT(*)
        
FROM Cadastro
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT * FROM cadastro
WHERE FPD IS NULL

In [0]:
%sql
SELECT * FROM cadastro
WHERE FPD IS NULL AND flag_mig2 IS NOT NULL

In [0]:
%sql
SELECT * FROM cadastro
WHERE FPD IS NULL AND FLAG_INSTALACAO != 0

In [0]:
%sql
SELECT * FROM cadastro
WHERE FPD IS NOT NULL AND FLAG_INSTALACAO != 1

In [0]:
%sql
SELECT Safra,
      FPD,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT Safra,
      PROD,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT Safra,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL AND PROD = "CMV"
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT Safra,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL AND flag_mig2 = "PRE"
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT Safra,
      ROUND(SUM(FPD)/COUNT(*),4) AS pct_FPD,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT Safra,
      PROD,
      ROUND(SUM(FPD)/COUNT(*),4) AS pct_target_prod,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT Safra,
      flag_mig2,
      ROUND(SUM(FPD)/COUNT(*),4) AS pct_target,  
      COUNT(*)
        
FROM Cadastro
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Verificar se o mesmo CPF tem mais de um produto e/ou flag_mig2
SELECT 
  NUM_CPF,
  COUNT(DISTINCT PROD) AS num_produtos,
  COUNT(DISTINCT flag_mig2) AS num_flag_mig2
FROM Cadastro
GROUP BY NUM_CPF
HAVING num_produtos > 1 OR num_flag_mig2 > 1

In [0]:
%sql
-- quantidade de registros totais e quantidade de CPF únicos
SELECT 
  COUNT(DISTINCT NUM_CPF) AS num_cpf_unicos,
  COUNT(*) AS qtd_registros
FROM Cadastro

# 2026 Michael


In [0]:
%sql
-- Unicidade por num_cpf+safra
SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT CONCAT(NUM_CPF, '#', SAFRA)) AS chaves_unicas_cpf_safra,
  COUNT(*) - COUNT(DISTINCT CONCAT(NUM_CPF, '#', SAFRA)) AS linhas_duplicadas
FROM cadastro;

In [0]:
%sql
-- verificação de Nulos/vazios nas colunas “core” (chave/tempo/target/metadados)
SELECT
  COUNT(*) AS total_linhas,

  SUM(CASE WHEN NUM_CPF IS NULL OR TRIM(NUM_CPF)='' THEN 1 ELSE 0 END) AS null_num_cpf,
  SUM(CASE WHEN SAFRA   IS NULL OR TRIM(SAFRA)=''   THEN 1 ELSE 0 END) AS null_safra,

  SUM(CASE WHEN FPD IS NULL OR TRIM(FPD)='' THEN 1 ELSE 0 END) AS null_fpd,
  SUM(CASE WHEN PROD IS NULL OR TRIM(PROD)='' THEN 1 ELSE 0 END) AS null_prod,
  SUM(CASE WHEN flag_mig2 IS NULL OR TRIM(flag_mig2)='' THEN 1 ELSE 0 END) AS null_flag_mig2,
  SUM(CASE WHEN FLAG_INSTALACAO IS NULL OR TRIM(FLAG_INSTALACAO)='' THEN 1 ELSE 0 END) AS null_flag_instalacao,

  SUM(CASE WHEN STATUSRF IS NULL OR TRIM(STATUSRF)='' THEN 1 ELSE 0 END) AS null_statusrf,
  SUM(CASE WHEN DATADENASCIMENTO IS NULL OR TRIM(DATADENASCIMENTO)='' THEN 1 ELSE 0 END) AS null_datadenascimento,
  SUM(CASE WHEN CEP_3_digitos IS NULL OR TRIM(CEP_3_digitos)='' THEN 1 ELSE 0 END) AS null_cep_3
FROM cadastro;

In [0]:
%sql
SELECT
  COUNT(*) AS total,

  SUM(CASE WHEN DATADENASCIMENTO IS NULL OR TRIM(DATADENASCIMENTO)='' THEN 1 ELSE 0 END) AS null_dn,
  SUM(CASE WHEN DATADENASCIMENTO IS NOT NULL AND TRIM(DATADENASCIMENTO)<>'' AND to_date(DATADENASCIMENTO,'dd/MM/yyyy') IS NULL THEN 1 ELSE 0 END) AS invalid_dn,

  SUM(CASE WHEN var_12 IS NULL OR TRIM(var_12)='' THEN 1 ELSE 0 END) AS null_var12,
  SUM(CASE WHEN var_12 IS NOT NULL AND TRIM(var_12)<>'' AND to_date(var_12,'dd/MM/yyyy') IS NULL THEN 1 ELSE 0 END) AS invalid_var12,

  SUM(CASE WHEN var_13 IS NULL OR TRIM(var_13)='' THEN 1 ELSE 0 END) AS null_var13,
  SUM(CASE WHEN var_13 IS NOT NULL AND TRIM(var_13)<>'' AND to_date(var_13,'dd/MM/yyyy') IS NULL THEN 1 ELSE 0 END) AS invalid_var13
FROM cadastro;

In [0]:
%sql
SELECT
  MIN(floor(months_between(to_date(concat(SAFRA,'01'),'yyyyMMdd'), to_date(DATADENASCIMENTO,'dd/MM/yyyy'))/12)) AS min_idade,
  MAX(floor(months_between(to_date(concat(SAFRA,'01'),'yyyyMMdd'), to_date(DATADENASCIMENTO,'dd/MM/yyyy'))/12)) AS max_idade
FROM cadastro
WHERE DATADENASCIMENTO IS NOT NULL AND TRIM(DATADENASCIMENTO)<>'' AND to_date(DATADENASCIMENTO,'dd/MM/yyyy') IS NOT NULL;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN var_03 IS NOT NULL AND TRIM(var_03)<>'' AND TRY_CAST(var_03 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_03,
  SUM(CASE WHEN var_04 IS NOT NULL AND TRIM(var_04)<>'' AND TRY_CAST(var_04 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_04,
  SUM(CASE WHEN var_05 IS NOT NULL AND TRIM(var_05)<>'' AND TRY_CAST(var_05 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_05,
  SUM(CASE WHEN var_06 IS NOT NULL AND TRIM(var_06)<>'' AND TRY_CAST(var_06 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_06,
  SUM(CASE WHEN var_07 IS NOT NULL AND TRIM(var_07)<>'' AND TRY_CAST(var_07 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_07,
  SUM(CASE WHEN var_08 IS NOT NULL AND TRIM(var_08)<>'' AND TRY_CAST(var_08 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_08,
  SUM(CASE WHEN var_09 IS NOT NULL AND TRIM(var_09)<>'' AND TRY_CAST(var_09 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_09,
  SUM(CASE WHEN var_15 IS NOT NULL AND TRIM(var_15)<>'' AND TRY_CAST(var_15 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_15,
  SUM(CASE WHEN var_16 IS NOT NULL AND TRIM(var_16)<>'' AND TRY_CAST(var_16 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_16,
  SUM(CASE WHEN var_17 IS NOT NULL AND TRIM(var_17)<>'' AND TRY_CAST(var_17 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_17,
  SUM(CASE WHEN var_22 IS NOT NULL AND TRIM(var_22)<>'' AND TRY_CAST(var_22 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_22,
  SUM(CASE WHEN var_23 IS NOT NULL AND TRIM(var_23)<>'' AND TRY_CAST(var_23 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_23,
  SUM(CASE WHEN var_24 IS NOT NULL AND TRIM(var_24)<>'' AND TRY_CAST(var_24 AS DOUBLE) IS NULL THEN 1 ELSE 0 END) AS nao_numericos_var_24
FROM cadastro;

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_07
SELECT
  MIN(CAST(var_07 AS DOUBLE)) AS min_var_07,
  MAX(CAST(var_07 AS DOUBLE)) AS max_var_07,
  AVG(CAST(var_07 AS DOUBLE)) AS avg_var_07,
  percentile_approx(CAST(var_07 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_07
FROM cadastro
WHERE var_07 IS NOT NULL AND TRIM(var_07)<>'' AND TRY_CAST(var_07 AS DOUBLE) IS NOT NULL;

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_08
SELECT
  MIN(CAST(var_08 AS DOUBLE)) AS min_var_08,
  MAX(CAST(var_08 AS DOUBLE)) AS max_var_08,
  AVG(CAST(var_08 AS DOUBLE)) AS avg_var_08,
  percentile_approx(CAST(var_08 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_08
FROM cadastro
WHERE var_08 IS NOT NULL AND TRIM(var_08)<>'' AND TRY_CAST(var_08 AS DOUBLE) IS NOT NULL;
    

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_09
SELECT
  MIN(CAST(var_09 AS DOUBLE)) AS min_var_09,
  MAX(CAST(var_09 AS DOUBLE)) AS max_var_09,
  AVG(CAST(var_09 AS DOUBLE)) AS avg_var_09,
  percentile_approx(CAST(var_09 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_09
FROM cadastro
WHERE var_09 IS NOT NULL AND TRIM(var_09)<>'' AND TRY_CAST(var_09 AS DOUBLE) IS NOT NULL;
    

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_10
SELECT
  MIN(CAST(var_10 AS DOUBLE)) AS min_var_10,
  MAX(CAST(var_10 AS DOUBLE)) AS max_var_10,
  AVG(CAST(var_10 AS DOUBLE)) AS avg_var_10,
  percentile_approx(CAST(var_10 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_10
FROM cadastro
WHERE var_10 IS NOT NULL AND TRIM(var_10)<>'' AND TRY_CAST(var_10 AS DOUBLE) IS NOT NULL;

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_11
SELECT
  MIN(CAST(var_11 AS DOUBLE)) AS min_var_11,
  MAX(CAST(var_11 AS DOUBLE)) AS max_var_11,
  AVG(CAST(var_11 AS DOUBLE)) AS avg_var_11,
  percentile_approx(CAST(var_11 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_11
FROM cadastro
WHERE var_11 IS NOT NULL AND TRIM(var_11)<>'' AND TRY_CAST(var_11 AS DOUBLE) IS NOT NULL;

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_12
SELECT
  MIN(CAST(var_12 AS DOUBLE)) AS min_var_12,
  MAX(CAST(var_12 AS DOUBLE)) AS max_var_12,
  AVG(CAST(var_12 AS DOUBLE)) AS avg_var_12,
  percentile_approx(CAST(var_12 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_12
FROM cadastro
WHERE var_12 IS NOT NULL AND TRIM(var_12)<>'' AND TRY_CAST(var_12 AS DOUBLE) IS NOT NULL;

In [0]:
%sql
--  Estatísticas/percentis para variáveis numéricas (mínimo para “Data Quality”) var_13
SELECT
  MIN(CAST(var_13 AS DOUBLE)) AS min_var_13,
  MAX(CAST(var_13 AS DOUBLE)) AS max_var_13,
  AVG(CAST(var_13 AS DOUBLE)) AS avg_var_13,
  percentile_approx(CAST(var_13 AS DOUBLE), array(0.01,0.05,0.50,0.95,0.99), 10000) AS pctl_var_13
FROM cadastro
WHERE var_13 IS NOT NULL AND TRIM(var_13)<>'' AND TRY_CAST(var_13 AS DOUBLE) IS NOT NULL;

In [0]:
%sql
-- Cardinalidade das colunas categóricas (para decidir encoding e limpeza)
SELECT
  STATUSRF,
  COUNT(*) AS qtd
FROM cadastro
GROUP BY STATUSRF
ORDER BY qtd DESC;

# Data Quality Report — `base_dados_cadastrais` (`cadastro`)

## 1) Objetivo
Registrar evidências de qualidade dos dados do **Cadastro**, focando em:
- **unicidade** do grão `NUM_CPF + SAFRA`
- **completude** de colunas core (chave/tempo/target/metadados)
- **qualidade de datas** (nascimento e possíveis datas em `var_*`)
- **tipagem real** (numéricas vs categóricas)
- **ranges e outliers** em variáveis numéricas principais

---

## 2) Unicidade do grão (CPF + SAFRA)

### 2.1) Evidência (resultado)
- **total_linhas:** 3.900.378  
- **chaves_unicas_cpf_safra:** 3.900.378  
- **linhas_duplicadas:** 0

### 2.2) Conclusão
A base está em **grão 1:1 por `NUM_CPF + SAFRA`** (join seguro no spine).

---

## 3) Completude (nulos/vazios) — colunas core

### 3.1) Evidência (resultado)
- `NUM_CPF`: 0 nulos/vazios
- `SAFRA`: 0 nulos/vazios
- `FPD`: 1.203.757 nulos/vazios (**volume alto**)
- `PROD`: 0 nulos/vazios
- `flag_mig2`: 1.266.478 nulos/vazios (**volume alto**)
- `FLAG_INSTALACAO`: 0 nulos/vazios
- `STATUSRF`: 15.154 nulos/vazios
- `DATADENASCIMENTO`: 16.831 nulos/vazios
- `CEP_3_digitos`: 292.051 nulos/vazios

### 3.2) Conclusão
- `FPD` e `flag_mig2` possuem **missing relevante** (impacta treino se usarem essa base como fonte de label).
- `CEP_3_digitos` possui missing moderado (recomendado criar flag).
- `DATADENASCIMENTO` tem missing baixo (boa candidata a feature via idade).

---

## 4) Sanity check — Idade na safra (derivada de `DATADENASCIMENTO`)
### 4.1) Evidência (resultado)
- **min_idade:** 4  
- **max_idade:** 131  

### 4.2) Interpretação
- Existem valores extremos (idade muito baixa e muito alta).
- Recomendação: manter a idade na Silver e criar flags/validações para outliers, com regra de negócio na Gold (ex.: cap/winsorize ou tratar fora do intervalo esperado).

---

## 5) Tipagem real das `var_*` (numérico vs categórico)

### 5.1) Evidência (resultado — contagem de não numéricos)
Para algumas colunas:
- `var_03` a `var_09`: 0 não numéricos (candidatas a numéricas)
- `var_15`: 585.003 não numéricos (candidata a categórica)
- `var_22`: 366.770 não numéricos (candidata a categórica/mista)
- `var_23`: 585.003 não numéricos (candidata a categórica)
- `var_24`: 2.410.043 não numéricos (forte candidata a categórica)

### 5.2) Conclusão
A base tem `var_*` **mistas**. Necessário:
- casting para numéricas apenas nas colunas com 0 não numéricos
- manter como string e padronizar nas colunas categóricas/mistas

---

## 6) Estatísticas de variáveis numéricas (amostra)

### 6.1) `var_07` (numérica, assimétrica com outliers)
Evidências:
- min: 0
- max: 37.975.401
- média: 46.166,7850
- percentis (p1/p5/p50/p95/p99): [0, 478.66, 2151.07, 142758, 277059]

Interpretação:
- distribuição altamente assimétrica (provável variável monetária/renda/valor).
- recomendação: considerar `log1p` e/ou caps na Gold.

### 6.2) `var_08` (numérica discreta)
- range: 1 a 98
- percentis: [21, 21, 32, 91, 92]

### 6.3) `var_09` (numérica discreta)
- range: 1 a 18
- percentis: [1, 4, 9, 9, 9]

### 6.4) `var_10` (numérica com concentração em valores baixos e cauda alta)
- range: 0 a 922.019
- percentis: [0, 0, 1, 3022, 922007]

### 6.5) `var_11` (numérica com valores negativos)
- min: -2791.67
- max: 49.329,24
- percentis: [0, 0, 2467.5, 13756.93, 22516.88]

Interpretação:
- existência de negativos sugere sentinela/ajuste/erro; recomendação: tratar negativos como inválidos ou criar flag.

---

## 7) Datas e parsing tolerante (erro identificado)

### 7.1) Problema observado
O parsing com `to_date(...,'dd/MM/yyyy')` falhou devido a valores inválidos como `'2807'`.

### 7.2) Recomendação
Usar funções tolerantes:
- `try_to_date(col,'dd/MM/yyyy')` (quando disponível)  
ou
- `try_cast(to_date(...))` dependendo do runtime.

**Query recomendada (para medir inválidos sem quebrar):**
```sql
SELECT
  COUNT(*) AS total,

  SUM(CASE WHEN DATADENASCIMENTO IS NULL OR TRIM(DATADENASCIMENTO)='' THEN 1 ELSE 0 END) AS null_dn,
  SUM(CASE WHEN DATADENASCIMENTO IS NOT NULL AND TRIM(DATADENASCIMENTO)<>'' AND try_to_date(DATADENASCIMENTO,'dd/MM/yyyy') IS NULL THEN 1 ELSE 0 END) AS invalid_dn,

  SUM(CASE WHEN var_12 IS NULL OR TRIM(var_12)='' THEN 1 ELSE 0 END) AS null_var12,
  SUM(CASE WHEN var_12 IS NOT NULL AND TRIM(var_12)<>'' AND try_to_date(var_12,'dd/MM/yyyy') IS NULL THEN 1 ELSE 0 END) AS invalid_var12

FROM cadastro;
```

## 8) Distribuição de STATUSRF (categoria)
### 8.1) Evidência (resultado)
- REGULAR: 3.848.697
- PENDENTE DE REGULARIZACAO: 31.217
- SUSPENSA: 2.689
- TITULAR FALECIDO: 2.398
- CANCELADA: 222
- NULA: 1
- null: 15.154

### 8.2) Conclusão
Variável categórica de baixa cardinalidade, adequada para encoding e com sinal potencial.

## 9) Recomendações de Data Quality (para automatizar)
- Validar SAFRA (YYYYMM) e não nulos de NUM_CPF e SAFRA.
- Monitorar % FPD nulo por safra (decidir regra de treino).
- Criar monitoramento de outliers em IDADE_ANOS, var_07, var_10 e negativos em var_11.
- Padronizar parsing de datas com funções tolerantes (try_to_date) e medir taxa de inválidos.